### Test Training

In [1]:
import importlib
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset as _TDS

import data.data, model.train
importlib.reload(data.data)
importlib.reload(model.train)

from data.data import DeforestationPatchDataset, DEFAULT_FEATURE_KEYS
from model.train import DANN_UNet, train_one_epoch

CACHE_DIR = Path("../cache_phase1")

# ─── 1. Pick a couple of tiles that actually have labels ──────────────────
train_paths = []
for p in sorted(CACHE_DIR.glob("*.npz")):
    with np.load(p, allow_pickle=False) as z:
        if "label" in z.files:
            train_paths.append(p)
    if len(train_paths) == 3:
        break
assert train_paths, "no cached tiles with labels"
print("train tiles:", [p.stem for p in train_paths])

# ─── 2. Build a MGRS-zone → region-id map (DANN head wants an int per sample) ─
zones = sorted({p.stem.split("_")[0] for p in train_paths})
region_map = {z: i for i, z in enumerate(zones)}
print("regions:", region_map)

class _WithRegion(_TDS):
    """Tacks a region_label onto each sample; the raw Dataset doesn't emit one."""
    def __init__(self, inner, region_map):
        self.inner = inner
        self.region_map = region_map
    def __len__(self): return len(self.inner)
    def __getitem__(self, idx):
        item = self.inner[idx]
        zone = item["tile"].split("_")[0]
        item["region_label"] = torch.tensor(self.region_map[zone], dtype=torch.long)
        return item

# ─── 3. Build a tiny dataloader ───────────────────────────────────────────
ds = _WithRegion(
    DeforestationPatchDataset(
        cache_paths=train_paths,
        feature_keys=DEFAULT_FEATURE_KEYS,
        patch_size=128, patches_per_tile=2,
        is_train=True, seed=0,
    ),
    region_map,
)
loader = DataLoader(ds, batch_size=2, shuffle=True, num_workers=0)

sample = ds[0]
C = sample["x"].shape[0]
print(f"in_channels = {C}   (post-Phase-2 feature stack)")

# ─── 4. Model + optim ─────────────────────────────────────────────────────
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"device: {device}")

net = DANN_UNet(num_regions=len(region_map), in_channels=C).to(device)
opt = torch.optim.AdamW(net.parameters(), lr=1e-4)

stats = train_one_epoch(
    model=net, loader=loader, optimizer=opt,
    device=device, epoch=0, num_epochs=1,
    alpha=0.1, spectral_aug_p=0.0,
)
print("stats:", stats)


train tiles: ['18NWG_6_6', '18NWH_1_4', '18NWJ_8_9']
regions: {'18NWG': 0, '18NWH': 1, '18NWJ': 2}


KeyError: 's2_pre_ndvi_p10'